# Q-테이블 히트맵 시각화 및 학습 결과 확인 (Gradio + Colab)

Q-테이블을 히트맵으로 시각화하고 학습 결과를 확인합니다.

**주요 기능**
- **히트맵을 통한 학습 결과 확인**: Q값 히트맵, 학습 곡선, 성공률
- **목표 상태로 향하는 경로 확인**: 4x4 그리드에 최적 정책 경로(파란 테두리) 표시
- **학습이 잘 된 영역 vs 부족한 영역 파악**: 녹색(잘 됨), 분홍(부족), 상태별 max Q 히트맵
- **하이퍼파라미터 조정 방향 설정**: 성공률·Q값 기반 자동 가이드

In [ ]:
# Google Colab 환경: 아래 주석을 해제하고 한 번만 실행
!pip install -q gradio gymnasium seaborn matplotlib

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import gradio as gr
import io
from PIL import Image

# Colab 등 한글 폰트가 없을 때: 나눔고딕 설치 후 matplotlib에 적용
def setup_korean_font():
    try:
        import subprocess
        subprocess.run(["apt-get", "update", "-qq"], check=True, capture_output=True)
        subprocess.run(["apt-get", "-qq", "-y", "install", "fonts-nanum"], check=True, capture_output=True)
        fm._load_fontmanager(try_read_cache=False)
        plt.rcParams["font.family"] = "NanumGothic"
        plt.rcParams["axes.unicode_minus"] = False  # 마이너스 기호 깨짐 방지
    except Exception:
        # 로컬 등에서 실패하면 기존 폰트 유지 (한글이 이미 되면 그대로 사용)
        pass

setup_korean_font()

In [ ]:
# FrozenLake 4x4 맵: S(0) F(1) F(2) F(3) / F(4) H(5) F(6) H(7) / F(8) F(9) F(10) H(11) / H(12) F(13) F(14) G(15)
HOLE_STATES = {5, 7, 11, 12}
GOAL_STATE = 15
START_STATE = 0

def run_qlearning_and_heatmap(n_episodes=2000, alpha=0.1, gamma=0.99):
    """Q-러닝 실행 후 히트맵, 경로/영역 시각화, 학습 결과, 가이드 반환 (Gradio용)."""
    env = gym.make("FrozenLake-v1", map_name="4x4")
    n_s, n_a = 16, 4
    Q = np.zeros((n_s, n_a))
    eps = 1.0
    returns_list = []

    for _ in range(n_episodes):
        s, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            a = env.action_space.sample() if np.random.random() < eps else np.argmax(Q[s])
            s2, r, term, trunc, _ = env.step(a)
            done = term or trunc
            Q[s, a] += alpha * (r + gamma * Q[s2].max() - Q[s, a])
            s = s2
            total_reward += r
        returns_list.append(total_reward)
        eps = max(0.01, eps * 0.995)

    # 학습 결과 정량화
    train_success_last100 = np.mean(returns_list[-100:]) if len(returns_list) >= 100 else np.mean(returns_list)
    test_successes = 0
    for _ in range(100):
        s, _ = env.reset()
        done = False
        while not done:
            a = np.argmax(Q[s])
            s, r, term, trunc, _ = env.step(a)
            done = term or trunc
            if term and r > 0:
                test_successes += 1
                break
    test_success_rate = test_successes / 100
    env.close()

    optimal_actions = np.argmax(Q, axis=1)
    actions_str = ["왼쪽", "아래", "오른쪽", "위"]
    max_q_per_state = Q.max(axis=1)

    # --- 시각화: 2x2 서브플롯 ---
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    # 1) Q-테이블 히트맵
    sns.heatmap(Q, annot=True, fmt=".2f", cmap="YlOrRd", ax=axes[0, 0],
                xticklabels=["왼", "아래", "오른", "위"], yticklabels=range(16))
    axes[0, 0].set_xlabel("행동")
    axes[0, 0].set_ylabel("상태")
    axes[0, 0].set_title("Q-테이블 히트맵 (밝을수록 좋은 행동)")

    # 2) 목표로 향하는 경로 + 학습 영역 (4x4 그리드)
    ax = axes[0, 1]
    # 행동 → 화살표 오프셋 (col, row): 왼(-1,0), 아래(0,1), 오른(1,0), 위(0,-1)
    arrow_dxy = {0: (-1, 0), 1: (0, 1), 2: (1, 0), 3: (0, -1)}
    labels = ["←", "↓", "→", "↑"]

    path_states = []
    s = START_STATE
    visited = set()
    while s != GOAL_STATE and s not in visited and len(path_states) < 20:
        visited.add(s)
        path_states.append(s)
        if s in HOLE_STATES:
            break
        a = optimal_actions[s]
        # FrozenLake 전이: 결정적 이동 (4x4 그리드)
        row, col = s // 4, s % 4
        dr, dc = arrow_dxy[a]
        nrow, ncol = max(0, min(3, row + dr)), max(0, min(3, col + dc))
        s = nrow * 4 + ncol
    if s == GOAL_STATE:
        path_states.append(s)

    for i in range(16):
        row, col = i // 4, i % 4
        # 학습 영역 색상: 잘 됨(녹색), 부족(빨강), 보통(노랑), 구멍/목표(회색/금색)
        if i in HOLE_STATES:
            color = "#666666"
        elif i == GOAL_STATE:
            color = "#FFD700"
        else:
            mq = max_q_per_state[i]
            if mq >= 0.3:
                color = "#90EE90"  # 잘 학습됨
            elif mq < 0.1:
                color = "#FFB6C1"  # 학습 부족
            else:
                color = "#FFFFE0"  # 보통
        ax.add_patch(plt.Rectangle((col, 3 - row), 1, 1, facecolor=color, edgecolor="black"))
        if i in path_states:
            ax.add_patch(plt.Rectangle((col, 3 - row), 1, 1, fill=False, edgecolor="blue", linewidth=3))
        if i in HOLE_STATES:
            ax.text(col + 0.5, 3 - row + 0.5, "H", ha="center", va="center", fontsize=14)
        elif i == GOAL_STATE:
            ax.text(col + 0.5, 3 - row + 0.5, "G", ha="center", va="center", fontsize=14)
        elif i == START_STATE:
            ax.text(col + 0.5, 3 - row + 0.5, "S", ha="center", va="center", fontsize=14)
        else:
            a = optimal_actions[i]
            ax.text(col + 0.5, 3 - row + 0.5, labels[a], ha="center", va="center", fontsize=12)
    ax.set_xlim(0, 4)
    ax.set_ylim(0, 4)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.set_title("경로(파란 테두리) / 녹색=잘 학습, 분홍=부족")
    ax.set_xticks([0.5, 1.5, 2.5, 3.5])
    ax.set_yticks([0.5, 1.5, 2.5, 3.5])
    ax.set_xticklabels([0, 1, 2, 3])
    ax.set_yticklabels([3, 2, 1, 0])

    # 3) 학습 곡선
    axes[1, 0].plot(returns_list, alpha=0.5, label="보상")
    window = min(100, max(1, len(returns_list) // 5))
    if len(returns_list) >= window:
        smoothed = np.convolve(returns_list, np.ones(window) / window, mode="valid")
        axes[1, 0].plot(range(window - 1, len(returns_list)), smoothed, "r-", linewidth=2, label="이동평균")
    axes[1, 0].set_xlabel("에피소드")
    axes[1, 0].set_ylabel("보상")
    axes[1, 0].set_title("학습 곡선 (목표 도달=1)")
    axes[1, 0].legend()
    axes[1, 0].set_ylim(-0.1, 1.1)

    # 4) 학습 영역 히트맵 (상태별 max Q)
    max_q_grid = np.array([max_q_per_state[r * 4 + c] for r in range(4) for c in range(4)]).reshape(4, 4)
    sns.heatmap(max_q_grid, annot=True, fmt=".2f", cmap="RdYlGn", ax=axes[1, 1], vmin=0, vmax=1)
    axes[1, 1].set_title("상태별 최대 Q값 (녹색=잘 학습, 빨강=부족)")
    axes[1, 1].set_xlabel("열")
    axes[1, 1].set_ylabel("행")

    plt.tight_layout()
    buf = io.BytesIO()
    plt.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    img = Image.open(buf).copy()
    buf.close()

    # --- 텍스트: 최적 행동 + 학습 결과 + 하이퍼파라미터 가이드 ---
    lines = [
        "=== 학습 결과 ===",
        f"· 마지막 100 에피소드 성공률: {train_success_last100:.1%}",
        f"· 테스트 100회 성공률: {test_success_rate:.1%}",
        "",
        "=== 상태별 최적 행동 ===",
    ]
    for s in range(16):
        lines.append(f"  상태 {s:2d}: {actions_str[optimal_actions[s]]}")
    lines.extend([
        "",
        "=== 하이퍼파라미터 조정 가이드 ===",
    ])
    # 가이드 로직
    if test_success_rate < 0.5:
        lines.append("· 성공률이 낮음 → 에피소드 수 증가(예: 4000~5000) 권장")
    if np.mean(max_q_per_state) < 0.2:
        lines.append("· Q값이 전반적으로 낮음 → 에피소드 수 증가 또는 alpha 0.15~0.2 시도")
    if (max_q_per_state < 0.1).sum() > 8:
        lines.append("· 학습 부족한 상태가 많음 → 에피소드 수 증가, epsilon_decay 완만하게")
    if test_success_rate >= 0.7 and np.mean(max_q_per_state) > 0.3:
        lines.append("· 학습이 양호함. gamma를 0.95로 낮추면 단기 보상에 더 민감해질 수 있음.")
    if alpha > 0.2:
        lines.append("· alpha가 큼 → 학습 불안정할 수 있음. 0.05~0.1 권장")
    if test_success_rate < 0.9 and not any("gamma" in ln for ln in lines):
        lines.append("· gamma 0.99 유지 시 장기 보상 중시. 0.95로 낮추면 단기 탐색 개선 가능")
    text = "\n".join(lines)
    return img, text

In [ ]:
def gradio_fn(n_episodes, alpha, gamma):
    """Gradio에서 호출: (n_episodes, alpha, gamma) → (이미지, 텍스트)."""
    n_episodes = int(n_episodes)
    img, text = run_qlearning_and_heatmap(n_episodes=n_episodes, alpha=alpha, gamma=gamma)
    return img, text

In [ ]:
# Gradio 앱: Colab에서는 아래 셀 실행 후 출력된 링크를 클릭하거나 인라인 위젯으로 사용
demo = gr.Interface(
    fn=gradio_fn,
    inputs=[
        gr.Number(value=2000, label="에피소드 수 (n_episodes)", minimum=100, maximum=10000, step=100),
        gr.Slider(0.01, 0.5, value=0.1, label="학습률 (alpha)", step=0.01),
        gr.Slider(0.5, 0.99, value=0.99, label="할인율 (gamma)", step=0.01),
    ],
    outputs=[
        gr.Image(label="히트맵 / 경로·학습영역 / 학습곡선"),
        gr.Textbox(label="학습 결과 및 하이퍼파라미터 가이드", lines=30),
    ],
    title="Q-테이블 히트맵 (FrozenLake 4x4)",
    description="히트맵, 목표 경로, 학습 영역(잘 됨/부족), 학습 곡선, 하이퍼파라미터 조정 가이드를 확인하세요.",
)

# Google Colab: share=False 로 실행하면 Colab 출력 영역에 앱이 표시됩니다.
demo.launch(share=False)